# Contact classifier AOT export

This notebook exports the trained contact classifier checkpoint to a Torch 2.x AOTInductor `.pt2` artifact.

You can choose between:
- **Dynamic batch size**: one exported program that accepts variable batch sizes.
- **Static batch size**: export specialized to a fixed batch size.

The heavy lifting is implemented in `export_contact_classifier.py` so this notebook stays minimal and reproducible.

In [5]:
import os
from pathlib import Path

import torch

from export_contact_classifier import export_contact_classifier_aot

project_root = Path("..").resolve()
print(f"Project root: {project_root}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Project root: /orcd/home/002/mvdokh/Deep-Learning/Contact Classification


device(type='cuda')

## Configure paths

Update these if you move the checkpoint or want to change the export location/name.

In [6]:
checkpoint_path = project_root / "Training" / "checkpoints" / "contact_classifier_WORKING.pt"
export_dir = project_root / "Export" / "artifacts"
export_dir.mkdir(parents=True, exist_ok=True)

export_name_dynamic = "contact_classifier_dynamic.pt2"
export_name_static = "contact_classifier_static_bs1.pt2"

print("Checkpoint:", checkpoint_path)
print("Export dir:", export_dir)

Checkpoint: /orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Training/checkpoints/contact_classifier_WORKING.pt
Export dir: /orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Export/artifacts


## Choose export mode

Set `USE_DYNAMIC_BATCH` to `True` for a single exported program that accepts a range of batch sizes.

If `False`, a static batch size export will be created instead.

In [7]:
USE_DYNAMIC_BATCH = True # set to False for static batch export

IMG_SIZE = 256
CHANNELS = 3

# Static batch configuration
STATIC_BATCH_SIZE = 1

# Dynamic batch configuration
DYNAMIC_MIN_BATCH = 1
DYNAMIC_MAX_BATCH = 128

## Run export

This cell calls the helper in `export_contact_classifier.py` and writes the `.pt2` file.

In [8]:
if not checkpoint_path.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

if USE_DYNAMIC_BATCH:
    export_path = export_dir / export_name_dynamic
    print("Exporting with dynamic batch dimension →", export_path)
    out_path = export_contact_classifier_aot(
        checkpoint_path=checkpoint_path,
        export_path=export_path,
        device=device,
        img_size=IMG_SIZE,
        channels=CHANNELS,
        dynamic_batch=True,
        dynamic_min_batch=DYNAMIC_MIN_BATCH,
        dynamic_max_batch=DYNAMIC_MAX_BATCH,
    )
else:
    export_path = export_dir / export_name_static
    print("Exporting with static batch size →", export_path)
    out_path = export_contact_classifier_aot(
        checkpoint_path=checkpoint_path,
        export_path=export_path,
        device=device,
        img_size=IMG_SIZE,
        channels=CHANNELS,
        static_batch_size=STATIC_BATCH_SIZE,
        dynamic_batch=False,
    )

print("Saved exported program to:", out_path)
out_path

Exporting with dynamic batch dimension → /orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Export/artifacts/contact_classifier_dynamic.pt2


/home/mvdokh/.conda/envs/keras-pytorch-2.9/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


Saved exported program to: /orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Export/artifacts/contact_classifier_dynamic.pt2


PosixPath('/orcd/home/002/mvdokh/Deep-Learning/Contact Classification/Export/artifacts/contact_classifier_dynamic.pt2')